# Handling Missing Values

Real-world datasets often include missing values (empty fields, `NaN`, `None`). How you handle them depends on *why* they’re missing and how much information is affected.

This notebook uses `../Datasets/missing_data.csv`, where some fields are blank (e.g., Category/Sub-Category/Profit/Quantity). When loaded with `pandas.read_csv`, those blank entries are treated as missing (`NaN`).

## Goals
- Detect missing values and quantify them per column/row
- Decide whether to drop, fill, or interpolate based on context
- Apply a consistent handling strategy
- Validate that the result still makes sense for analysis/modeling

## Quick detection toolkit (pandas)
```python
df.isna().sum().sort_values(ascending=False)     # missing count per column
df.isna().mean().mul(100).round(2)               # missing % per column
df[df.isna().any(axis=1)].head()                 # rows with at least one missing value
```

## Common strategies
**1) Drop missing**
```python
df_drop_rows = df.dropna()                             # drop rows with any missing values
df_drop_cols = df.dropna(axis=1, thresh=len(df)*0.8)   # keep cols with >=80% non-missing
```

**2) Simple imputation (`fillna`)**
```python
df2 = df.copy()
df2["Profit"] = df2["Profit"].fillna(df2["Profit"].median())
df2["Category"] = df2["Category"].fillna(df2["Category"].mode().iat[0])
```

**3) Forward/Backward fill (often for ordered/time-like data)**
```python
df2 = df.sort_values("Customer ID").copy()
df2["Sub-Category"] = df2["Sub-Category"].ffill().bfill()
```

**4) Group-wise imputation (keeps structure)**
```python
df2 = df.copy()
df2["Profit"] = df2["Profit"].fillna(df2.groupby("Category")["Profit"].transform("median"))
```

## After handling: re-check + sanity check
```python
df2.isna().sum()
df2.describe(include="all")
```

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
df = pd.read_csv('../Datasets/missing_data.csv')

In [4]:
df.head(20)

,Customer ID,Customer Name,Region,Category,Sub-Category,Sales,Profit,Quantity,Avg Discount
0,AA-10315,Alex Avila,Central,Office Supplies,Appliances,362.94,90.74,3.0,0.0
1,AA-10315,Alex Avila,Central,Office Supplies,Binders,11.54,5.77,2.0,0.0
2,AA-10315,Alex Avila,Central,NaN,Fasteners,2.30,0.78,1.0,0.2
3,AA-10315,Alex Avila,Central,Office Supplies,Supplies,3930.07,NaN,3.0,0.2
4,AA-10315,Alex Avila,Central,Technology,NaN,41.72,5.74,7.0,0.2
5,AA-10315,Alex Avila,Central,Technology,Phones,431.98,32.40,NaN,0.2
6,AA-10315,Alex Avila,East,Furniture,Furnishings,14.56,6.26,2.0,0.0
7,AA-10315,Alex Avila,East,Office Supplies,Paper,14.94,7.02,3.0,0.0
8,AA-10315,Alex Avila,West,NaN,Appliances,52.98,14.83,2.0,0.0
9,AA-10315,Alex Avila,West,Office Supplies,Binders,673.57,252.59,2.0,0.2
